In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

## Settings

In [ ]:
settings = {
    "image_path": "<PATH_TO_MEASUREMENT>",
    "image_size": "<IMAGE_SIDE_PIXELS>",
    "tile_size": "<CORE_SIDE_PIXELS>",
    "overlap_fraction": "<SHARED_BAND_AS_FRACTION_OF_CORE_WIDTH>",
    "context_pixels": "<CONTEXT_PIXELS_PER_SIDE>",
    "wavelength_m": "<WAVELENGTH_IN_METRES>",
    "pixel_pitch_m": "<PIXEL_PITCH_IN_METRES>",
    "z_min_mm": "<MINIMUM_PROPAGATION_DISTANCE_IN_MM>",
    "z_max_mm": "<MAXIMUM_PROPAGATION_DISTANCE_IN_MM>",
    "propagation_pad_factor": "<FFT_PADDING_FACTOR>",
    "population": "<GP_POPULATION_SIZE>",
    "generations": "<GP_GENERATIONS>",
    "crossover_probability": "<CROSSOVER_PROBABILITY>",
    "mutation_probability": "<MUTATION_PROBABILITY>",
    "max_tree_height": "<MAXIMUM_TREE_HEIGHT>",
    "gp_min_iterations": "<MINIMUM_SOLVER_ITERATIONS>",
    "gp_max_iterations": "<MAXIMUM_SOLVER_ITERATIONS>",
    "lr_amplitude_bounds": ("<MIN_AMPLITUDE_PHASE_RATE>", "<MAX_AMPLITUDE_PHASE_RATE>"),
    "lr_z_bounds": ("<MIN_DISTANCE_RATE>", "<MAX_DISTANCE_RATE>"),
    "lambda_tv_bounds": ("<MIN_TV_WEIGHT>", "<MAX_TV_WEIGHT>"),
    "lambda_binary_bounds": ("<MIN_BINARY_WEIGHT>", "<MAX_BINARY_WEIGHT>"),
    "lambda_contrast_magnitude_bounds": ("<MIN_CONTRAST_MAGNITUDE>", "<MAX_CONTRAST_MAGNITUDE>"),
    "delta_huber_bounds": ("<MIN_HUBER_THRESHOLD>", "<MAX_HUBER_THRESHOLD>"),
    "sample_interval": "<SSIM_CHECK_INTERVAL_IN_ITERATIONS>",
    "early_stopping_min_iterations": "<EARLIEST_STOPPING_ITERATION>",
    "early_stopping_patience": "<STOPPING_PATIENCE_IN_ITERATIONS>",
    "early_stopping_min_delta": "<MINIMUM_SSIM_IMPROVEMENT>",
    "seed": "<RANDOM_SEED>",
    "evolution_iteration_cap": None,
    "initial_phase_rad": "<Initialization>",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}


## Inner reconstruction solver

In [ ]:
from __future__ import annotations

import random
import numpy as np
import torch
from skimage.metrics import structural_similarity


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


def fftshift_torch(x: torch.Tensor) -> torch.Tensor:
    for dimension in (-2, -1):
        x = torch.roll(x, shifts=x.size(dimension) // 2, dims=dimension)
    return x


def ifftshift_torch(x: torch.Tensor) -> torch.Tensor:
    for dimension in (-2, -1):
        x = torch.roll(x, shifts=-(x.size(dimension) // 2), dims=dimension)
    return x


def angular_spectrum(
    field: torch.Tensor,
    z_m: torch.Tensor | float,
    cfg: Config,
    pad_factor: int | None = None,
) -> torch.Tensor:
    height, width = field.shape[-2:]
    factor = cfg.propagation_pad_factor if pad_factor is None else pad_factor
    pad = factor * max(height, width)
    padded_side = max(height, width) + 2 * pad
    device = field.device

    fx = torch.arange(
        -padded_side / 2, padded_side / 2, device=device, dtype=torch.float32
    ) / (cfg.pixel_pitch_m * padded_side)
    fy = torch.arange(
        -padded_side / 2, padded_side / 2, device=device, dtype=torch.float32
    ) / (cfg.pixel_pitch_m * padded_side)
    frequency_y, frequency_x = torch.meshgrid(fy, fx, indexing="ij")
    transfer = torch.exp(
        -1j
        * np.pi
        * cfg.wavelength_m
        * z_m
        * (frequency_x.square() + frequency_y.square())
    )

    boundary = torch.cat((field[0], field[-1], field[:, 0], field[:, -1]))
    fill_value = boundary.mean()
    padded = torch.ones(
        (padded_side, padded_side), dtype=field.dtype, device=device
    ) * fill_value
    padded[:height, :width] = field

    spectrum = fftshift_torch(torch.fft.fft2(padded))
    propagated = torch.fft.ifft2(ifftshift_torch(spectrum * transfer))
    return propagated[:height, :width]


def z_from_free(z_free: torch.Tensor, cfg: Config) -> torch.Tensor:
    span_m = (cfg.z_max_mm - cfg.z_min_mm) * 1e-3
    return torch.sigmoid(z_free) * span_m + cfg.z_min_mm * 1e-3


def free_from_z_mm(z_mm: float, cfg: Config) -> float:
    clipped = float(np.clip(z_mm, cfg.z_min_mm + 1e-6, cfg.z_max_mm - 1e-6))
    fraction = (clipped - cfg.z_min_mm) / (cfg.z_max_mm - cfg.z_min_mm)
    return float(np.log(fraction / (1.0 - fraction)))


def huber_loss(predicted: torch.Tensor, target: torch.Tensor, delta: float) -> torch.Tensor:
    residual = predicted - target
    absolute = residual.abs()
    return torch.where(
        absolute <= delta,
        0.5 * residual.square(),
        delta * (absolute - 0.5 * delta),
    ).mean()


def tv_loss(value: torch.Tensor) -> torch.Tensor:
    horizontal = value[:, 1:] - value[:, :-1]
    vertical = value[1:, :] - value[:-1, :]
    return horizontal.abs().mean() + vertical.abs().mean()


def binary_loss(value: torch.Tensor) -> torch.Tensor:
    return (4.0 * value * (1.0 - value)).mean()


def contrast_loss(value: torch.Tensor) -> torch.Tensor:
    return torch.std(value)


def normalise_01(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array, dtype=np.float64)
    lo, hi = float(np.min(array)), float(np.max(array))
    return ((array - lo) / max(hi - lo, 1e-12)).astype(np.float32)


def detector_ssim(reference: np.ndarray, prediction: np.ndarray) -> float:
    return float(
        structural_similarity(
            normalise_01(reference),
            normalise_01(prediction),
            data_range=1.0,
        )
    )


def run_gradient_reconstruction(
    image: np.ndarray,
    hp: dict[str, float | int],
    cfg: Config,
    seed: int,
) -> dict:
    set_seed(seed)
    device = torch.device(cfg.device)
    image = image.astype(np.float32)
    target_np = image / max(float(image.max()), 1e-8)
    target = torch.from_numpy(target_np).to(device)

    amplitude = torch.full_like(target, 0.5).requires_grad_(True)
    phase = torch.full_like(target, float(cfg.initial_phase_rad)).requires_grad_(True)
    z_free = torch.tensor(
        [free_from_z_mm(float(hp["z_init_mm"]), cfg)],
        dtype=torch.float32,
        device=device,
        requires_grad=True,
    )
    optimizer = torch.optim.Adam(
        [
            {
                "params": [amplitude, phase],
                "lr": float(hp["lr_amplitude"]),
            },
            {"params": [z_free], "lr": float(hp["lr_z"])},
        ]
    )
    n_iters = int(hp["n_iters"])
    best_state = None
    best_ssim = -float("inf")
    last_meaningful_improvement_iteration = 0

    for iteration in range(n_iters):
        optimizer.zero_grad()
        amp = amplitude.clamp(0.0, 1.0)
        field = amp * torch.exp(1j * phase)
        z_m = z_from_free(z_free, cfg)[0]
        propagated = angular_spectrum(field, z_m, cfg)
        predicted = propagated.abs().square()

        data_term = huber_loss(predicted, target, float(hp["delta_huber"]))
        zero = predicted.new_zeros(())
        tv_term = (
            float(hp["lambda_tv"]) * (tv_loss(amp) + tv_loss(phase))
            if float(hp["lambda_tv"]) != 0.0
            else zero
        )
        binary_term = (
            float(hp["lambda_binary"]) * binary_loss(amp)
            if float(hp["lambda_binary"]) != 0.0
            else zero
        )
        contrast_term = (
            float(hp["lambda_contrast"]) * contrast_loss(amp)
            if float(hp["lambda_contrast"]) != 0.0
            else zero
        )
        total = data_term + tv_term + binary_term + contrast_term
        total.backward()
        torch.nn.utils.clip_grad_norm_([z_free], max_norm=0.25)
        optimizer.step()
        with torch.no_grad():
            amplitude.clamp_(0.0, 1.0)
        del (
            amp, field, z_m, propagated, predicted,
            data_term, tv_term, binary_term, contrast_term, total, zero,
        )

        should_record = (
            iteration % cfg.sample_interval == 0 or iteration == n_iters - 1
        )
        if should_record:
            with torch.no_grad():
                amp_record = amplitude.clamp(0.0, 1.0)
                phase_record = phase
                z_record_m = z_from_free(z_free, cfg)[0]
                propagated_record = angular_spectrum(
                    amp_record * torch.exp(1j * phase_record),
                    z_record_m,
                    cfg,
                )
                predicted_record = propagated_record.abs().square()
                pred_np = predicted_record.detach().cpu().numpy()
                ssim_value = detector_ssim(target_np, pred_np)
                improvement = ssim_value - best_ssim
                if improvement > 0.0:
                    best_ssim = ssim_value
                    best_state = {
                        "amplitude": amp_record.detach().cpu().numpy().astype(
                            np.float32
                        ),
                        "phase": phase_record.detach().cpu().numpy().astype(
                            np.float32
                        ),
                        "predicted_intensity": pred_np.astype(np.float32),
                        "z_mm": float(z_record_m.detach().cpu()) * 1e3,
                    }
                if improvement > cfg.early_stopping_min_delta:
                    last_meaningful_improvement_iteration = iteration + 1
                del (
                    amp_record, phase_record, z_record_m,
                    propagated_record, predicted_record,
                )
            executed_iterations = iteration + 1
            if (
                executed_iterations >= cfg.early_stopping_min_iterations
                and executed_iterations - last_meaningful_improvement_iteration
                >= cfg.early_stopping_patience
            ):
                break

    if best_state is None:
        raise RuntimeError("The reconstruction did not produce a valid sampled state")
    return best_state


## Genetic programming

In [ ]:
import copy
import math
import operator
import random
from functools import partial

import numpy as np
from deap import base, creator, gp, tools


HYPERPARAMETER_NAMES = (
    "n_iters",
    "lr_amplitude",
    "lr_z",
    "lambda_tv",
    "lambda_binary",
    "lambda_contrast",
    "delta_huber",
    "z_init_mm",
)


def protected_division(first, second):
    return first / (1e-8 + abs(second))


def protected_sqrt(value):
    return math.sqrt(abs(value))


def protected_exp(value):
    return math.exp(max(min(value, 6.0), -6.0))


def protected_log1p(value):
    return math.log1p(max(value, -0.999999))


def build_primitive_set():
    primitive_set = gp.PrimitiveSet("HPGEN", 7)
    primitive_set.addPrimitive(operator.add, 2)
    primitive_set.addPrimitive(operator.sub, 2)
    primitive_set.addPrimitive(operator.mul, 2)
    primitive_set.addPrimitive(protected_division, 2)
    primitive_set.addPrimitive(min, 2)
    primitive_set.addPrimitive(max, 2)
    primitive_set.addPrimitive(abs, 1)
    primitive_set.addPrimitive(math.tanh, 1)
    primitive_set.addPrimitive(protected_log1p, 1)
    primitive_set.addPrimitive(protected_sqrt, 1)
    primitive_set.addPrimitive(protected_exp, 1)
    primitive_set.addEphemeralConstant(
        "rand", partial(random.uniform, -3.0, 3.0)
    )
    primitive_set.renameArguments(
        ARG0="cx",
        ARG1="cy",
        ARG2="mI",
        ARG3="sI",
        ARG4="gM",
        ARG5="gS",
        ARG6="bias",
    )
    return primitive_set


PSET = build_primitive_set()


def bounded_map(value, lower, upper):
    if not np.isfinite(value):
        value = 0.0
    logistic = 1.0 / (1.0 + math.exp(-float(np.clip(value, -60.0, 60.0))))
    return lower + (upper - lower) * logistic


def compile_policy(individual, cfg, iteration_cap=None):
    functions = [gp.compile(expr=tree, pset=PSET) for tree in individual[:8]]

    def generate(features):
        raw = []
        for function in functions:
            try:
                value = float(function(*features))
            except (ArithmeticError, OverflowError, ValueError):
                value = 0.0
            raw.append(value if np.isfinite(value) else 0.0)
        n_iters = int(
            round(
                bounded_map(
                    raw[0],
                    cfg.gp_min_iterations,
                    cfg.gp_max_iterations,
                )
            )
        )
        if iteration_cap is not None:
            n_iters = min(n_iters, iteration_cap)
        return {
            "n_iters": n_iters,
            "lr_amplitude": bounded_map(raw[1], *cfg.lr_amplitude_bounds),
            "lr_z": bounded_map(raw[2], *cfg.lr_z_bounds),
            "lambda_tv": bounded_map(raw[3], *cfg.lambda_tv_bounds),
            "lambda_binary": bounded_map(raw[4], *cfg.lambda_binary_bounds),
            "lambda_contrast": -bounded_map(
                raw[5], *cfg.lambda_contrast_magnitude_bounds
            ),
            "delta_huber": bounded_map(raw[6], *cfg.delta_huber_bounds),
            "z_init_mm": bounded_map(raw[7], cfg.z_min_mm, cfg.z_max_mm),
        }

    return generate


def make_deap_types():
    if not hasattr(creator, "USAFPolicyFitness"):
        creator.create(
            "USAFPolicyFitness",
            base.Fitness,
            weights=(1.0, -0.25, -0.02),
        )
    if not hasattr(creator, "USAFPolicyIndividual"):
        creator.create(
            "USAFPolicyIndividual",
            list,
            fitness=creator.USAFPolicyFitness,
        )


def quality_key(individual):
    return (
        float(individual.fitness.values[0]),
        -float(individual.fitness.values[1]),
        -float(individual.fitness.values[2]),
    )


def quality_first_select(individuals, k):
    return sorted(individuals, key=quality_key, reverse=True)[:k]


def tournament_parent(population, tournament_size=3):
    contestants = random.sample(
        list(population),
        min(tournament_size, len(population)),
    )
    return max(contestants, key=quality_key)


def breed_offspring(population, toolbox, cfg):
    offspring = []
    while len(offspring) < cfg.population:
        draw = random.random()
        if draw < cfg.crossover_probability:
            parent_a = tournament_parent(population)
            remaining_parents = [
                individual
                for individual in population
                if individual is not parent_a
            ]
            parent_b = tournament_parent(remaining_parents)
            child_a = copy.deepcopy(parent_a)
            child_b = copy.deepcopy(parent_b)
            child_a, child_b = toolbox.mate(child_a, child_b)
            child = child_a
            del child.fitness.values
        elif draw < cfg.crossover_probability + cfg.mutation_probability:
            child = copy.deepcopy(tournament_parent(population))
            child, = toolbox.mutate(child)
            del child.fitness.values
        else:
            child = copy.deepcopy(tournament_parent(population))
        offspring.append(child)
    return offspring


def evolve_policy(evaluate_callback, cfg):
    make_deap_types()
    set_seed(cfg.seed)
    toolbox = base.Toolbox()
    toolbox.register(
        "tree",
        gp.genHalfAndHalf,
        pset=PSET,
        min_=1,
        max_=3,
    )

    def create_individual():
        return creator.USAFPolicyIndividual(
            [
                gp.PrimitiveTree(toolbox.tree())
                for _ in range(len(HYPERPARAMETER_NAMES))
            ]
        )

    def mate(first, second):
        gene = random.randrange(len(HYPERPARAMETER_NAMES))
        gp.cxOnePoint(first[gene], second[gene])
        return first, second

    def mutate(individual):
        gene = random.randrange(len(HYPERPARAMETER_NAMES))
        expression = gp.genFull(pset=PSET, min_=0, max_=2)
        individual[gene], = gp.mutUniform(
            individual[gene], expr=lambda **_: expression, pset=PSET
        )
        return (individual,)

    toolbox.register("individual", create_individual)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("mate", mate)
    toolbox.register("mutate", mutate)
    toolbox.decorate(
        "mate",
        gp.staticLimit(
            key=lambda individual: max(tree.height for tree in individual),
            max_value=cfg.max_tree_height,
        ),
    )
    toolbox.decorate(
        "mutate",
        gp.staticLimit(
            key=lambda individual: max(tree.height for tree in individual),
            max_value=cfg.max_tree_height,
        ),
    )
    toolbox.register("select", quality_first_select)
    population = toolbox.population(n=cfg.population)
    fitness_cache = {}

    def evaluate_generation(individuals):
        invalid = [item for item in individuals if not item.fitness.valid]
        for individual in invalid:
            genome_signature = "|".join(str(tree) for tree in individual)
            if genome_signature in fitness_cache:
                individual.fitness.values = fitness_cache[genome_signature]
            else:
                individual.fitness.values = evaluate_callback(
                    individual,
                    cfg.seed,
                    cfg.evolution_iteration_cap,
                )
                fitness_cache[genome_signature] = tuple(
                    individual.fitness.values
                )

    evaluate_generation(population)
    population = toolbox.select(population, k=len(population))
    for _ in range(1, cfg.generations + 1):
        offspring = breed_offspring(population, toolbox, cfg)
        evaluate_generation(offspring)
        population = toolbox.select(population + offspring, k=cfg.population)
    best = max(population, key=quality_key)
    return [copy.deepcopy(tree) for tree in best]


## Contextual tiling and assembly

In [ ]:
from dataclasses import dataclass

import numpy as np


@dataclass
class Tile:
    index: int
    row: int
    column: int
    core_bounds: tuple
    retained_bounds: tuple
    input_bounds: tuple
    input_intensity: np.ndarray
    core_slice: tuple
    retained_slice: tuple


def iter_tiles(image, core_size, context_pixels, overlap_fraction):
    image = np.asarray(image)
    if image.ndim != 2 or image.shape[0] != image.shape[1]:
        raise ValueError("The detector image must be a two-dimensional square.")
    if core_size <= 0 or image.shape[0] % core_size:
        raise ValueError("The image side must be divisible by core_size.")
    if context_pixels < 0 or not np.isfinite(overlap_fraction) or overlap_fraction < 0:
        raise ValueError("Context and overlap must be nonnegative.")

    overlap_pixels = int(round(overlap_fraction * core_size))
    expansion_before = overlap_pixels // 2
    expansion_after = overlap_pixels - expansion_before
    retained_size = core_size + overlap_pixels
    core_start = context_pixels + expansion_before
    core_slice = (
        slice(core_start, core_start + core_size),
        slice(core_start, core_start + core_size),
    )
    retained_slice = (
        slice(context_pixels, context_pixels + retained_size),
        slice(context_pixels, context_pixels + retained_size),
    )
    height, width = image.shape
    grid_size = height // core_size
    for row in range(grid_size):
        for column in range(grid_size):
            y0, x0 = row * core_size, column * core_size
            y1, x1 = y0 + core_size, x0 + core_size
            ry0, ry1 = y0 - expansion_before, y1 + expansion_after
            rx0, rx1 = x0 - expansion_before, x1 + expansion_after
            iy0, iy1 = ry0 - context_pixels, ry1 + context_pixels
            ix0, ix1 = rx0 - context_pixels, rx1 + context_pixels
            sy0, sy1 = max(0, iy0), min(height, iy1)
            sx0, sx1 = max(0, ix0), min(width, ix1)
            computational_input = np.pad(
                image[sy0:sy1, sx0:sx1],
                ((sy0 - iy0, iy1 - sy1), (sx0 - ix0, ix1 - sx1)),
                mode="edge",
            )
            yield Tile(
                index=row * grid_size + column,
                row=row,
                column=column,
                core_bounds=(y0, y1, x0, x1),
                retained_bounds=(ry0, ry1, rx0, rx1),
                input_bounds=(iy0, iy1, ix0, ix1),
                input_intensity=computational_input,
                core_slice=core_slice,
                retained_slice=retained_slice,
            )


def blending_window(tile_size, overlap_fraction):
    if overlap_fraction == 0:
        return np.ones((tile_size, tile_size), dtype=np.float32)
    window = np.maximum(np.hanning(tile_size).astype(np.float32), 1e-3)
    return np.outer(window, window)


def assemble_tiles(tiles, amplitudes, phases, image_shape, overlap_fraction):
    amplitude_sum = np.zeros(image_shape, dtype=np.float64)
    phase_sum = np.zeros(image_shape, dtype=np.complex128)
    weight_sum = np.zeros(image_shape, dtype=np.float64)
    height, width = image_shape
    for tile, amplitude, phase in zip(tiles, amplitudes, phases, strict=True):
        amplitude, phase = np.asarray(amplitude), np.asarray(phase)
        if amplitude.shape != tile.input_intensity.shape or phase.shape != amplitude.shape:
            raise ValueError("Reconstructions must have the complete computational input shape.")
        retained_amplitude = amplitude[tile.retained_slice]
        retained_phase = phase[tile.retained_slice]
        window = blending_window(retained_amplitude.shape[0], overlap_fraction)
        y0, y1, x0, x1 = tile.retained_bounds
        sy0, sy1 = max(0, y0), min(height, y1)
        sx0, sx1 = max(0, x0), min(width, x1)
        source = (slice(sy0 - y0, sy1 - y0), slice(sx0 - x0, sx1 - x0))
        destination = (slice(sy0, sy1), slice(sx0, sx1))
        weights = window[source]
        amplitude_sum[destination] += weights * retained_amplitude[source]
        phase_sum[destination] += weights * np.exp(1j * retained_phase[source])
        weight_sum[destination] += weights
    if np.any(weight_sum == 0):
        raise ValueError("The supplied tiles do not cover the full output image.")
    return amplitude_sum / weight_sum, np.angle(phase_sum / weight_sum)


## Reconstruction pipeline

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import time

import imageio.v3 as iio


def make_configuration(values):
    missing = []
    for name, value in values.items():
        entries = value if isinstance(value, (tuple, list)) else (value,)
        if any(isinstance(item, str) and item.startswith("<") for item in entries):
            missing.append(name)
    if missing:
        raise ValueError("Fill the settings cell before running. Missing values: " + ", ".join(missing))

    cfg = SimpleNamespace(**values)
    integer_fields = (
        "image_size", "tile_size", "context_pixels", "propagation_pad_factor",
        "population", "generations", "max_tree_height", "gp_min_iterations",
        "gp_max_iterations", "sample_interval", "early_stopping_min_iterations",
        "early_stopping_patience", "seed",
    )
    for name in integer_fields:
        value = getattr(cfg, name)
        if isinstance(value, bool) or not isinstance(value, (int, np.integer)):
            raise ValueError(name + " must be an integer, without quotation marks.")
        if value < 0:
            raise ValueError(name + " must be nonnegative.")
    if min(cfg.image_size, cfg.tile_size, cfg.gp_min_iterations, cfg.sample_interval) <= 0:
        raise ValueError("Image size, core size, iteration budget and check interval must be positive.")
    if cfg.tile_size < 7 or cfg.image_size % cfg.tile_size:
        raise ValueError("The image side must be divisible by the core side, which must support the SSIM window.")
    if cfg.population < 2:
        raise ValueError("At least two policies are required for crossover.")
    if cfg.gp_max_iterations < cfg.gp_min_iterations:
        raise ValueError("Maximum iterations must not be below minimum iterations.")
    if not (0 <= cfg.overlap_fraction <= 1):
        raise ValueError("Overlap fraction must lie between zero and one.")
    if cfg.wavelength_m <= 0 or cfg.pixel_pitch_m <= 0 or not (0 < cfg.z_min_mm < cfg.z_max_mm):
        raise ValueError("Optical lengths must be positive and distance bounds must be ordered.")
    if min(cfg.crossover_probability, cfg.mutation_probability) < 0 or cfg.crossover_probability + cfg.mutation_probability > 1:
        raise ValueError("Variation probabilities must be nonnegative and sum to at most one.")
    for name in (
        "lr_amplitude_bounds", "lr_z_bounds", "lambda_tv_bounds",
        "lambda_binary_bounds", "lambda_contrast_magnitude_bounds", "delta_huber_bounds",
    ):
        lower, upper = getattr(cfg, name)
        if not (np.isfinite(lower) and np.isfinite(upper) and 0 <= lower <= upper):
            raise ValueError(name + " must contain two ordered nonnegative numeric bounds.")
    if cfg.evolution_iteration_cap is not None and (
        not isinstance(cfg.evolution_iteration_cap, int) or cfg.evolution_iteration_cap <= 0
    ):
        raise ValueError("The optional evolution iteration cap must be a positive integer or None.")
    if str(cfg.device).startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable.")
    return cfg


def load_measurement(cfg):
    path = Path(cfg.image_path)
    if not path.is_file():
        raise FileNotFoundError("Set image_path to an existing measurement file.")
    raw = iio.imread(path)
    if raw.ndim == 3:
        rgb = raw[..., :3].astype(np.float32)
        raw = 0.2126 * rgb[..., 0] + 0.7152 * rgb[..., 1] + 0.0722 * rgb[..., 2]
    if raw.ndim != 2 or raw.shape != (cfg.image_size, cfg.image_size):
        raise ValueError("The measurement must match image_size. No resizing is performed.")
    if np.issubdtype(raw.dtype, np.integer):
        image = raw.astype(np.float32) / float(np.iinfo(raw.dtype).max)
    else:
        image = raw.astype(np.float32)
        if not np.isfinite(image).all():
            raise ValueError("The measurement contains nonfinite values.")
        if image.min() < 0 or image.max() > 1:
            lo, hi = float(image.min()), float(image.max())
            image = (image - lo) / max(hi - lo, 1e-12)
    return np.clip(image, 0.0, 1.0)


def tile_features(tile, image):
    y0, y1, x0, x1 = tile.core_bounds
    return (
        0.5 * (x0 + x1) / image.shape[1],
        0.5 * (y0 + y1) / image.shape[0],
        float(tile.input_intensity.mean()),
        float(tile.input_intensity.std()),
        float(image.mean()),
        float(image.std()),
        1.0,
    )


def make_policy_evaluator(tiles, image, cfg):
    features = [tile_features(tile, image) for tile in tiles]

    def evaluate(individual, seed, iteration_cap):
        generate = compile_policy(individual, cfg, iteration_cap)
        scores, runtimes = [], []
        for tile, feature in zip(tiles, features):
            start = time.perf_counter()
            hp = generate(feature)
            result = run_gradient_reconstruction(tile.input_intensity, hp, cfg, seed + tile.index)
            target = tile.input_intensity[tile.core_slice] / max(float(tile.input_intensity.max()), 1e-8)
            prediction = result["predicted_intensity"][tile.core_slice]
            scores.append(detector_ssim(target, prediction))
            runtimes.append(time.perf_counter() - start)
        set_seed(seed + len(tiles) - 1)
        return float(np.mean(scores)), float(np.sum(runtimes)), sum(len(tree) for tree in individual[:8])

    return evaluate


def reconstruct_with_policy(tiles, image, policy, cfg):
    generate = compile_policy(policy, cfg)
    amplitudes, phases = [], []
    for tile in tiles:
        hp = generate(tile_features(tile, image))
        result = run_gradient_reconstruction(tile.input_intensity, hp, cfg, cfg.seed + tile.index)
        amplitudes.append(result["amplitude"])
        phases.append(result["phase"])
    return assemble_tiles(tiles, amplitudes, phases, image.shape, cfg.overlap_fraction)


## Run and display


In [ ]:
cfg = make_configuration(settings)
measurement = load_measurement(cfg)
tiles = list(iter_tiles(measurement, cfg.tile_size, cfg.context_pixels, cfg.overlap_fraction))
evaluate = make_policy_evaluator(tiles, measurement, cfg)
policy = evolve_policy(evaluate, cfg)
amplitude, phase = reconstruct_with_policy(tiles, measurement, policy, cfg)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(amplitude, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Amplitude")
axes[1].imshow(phase, cmap="gray", vmin=-np.pi, vmax=np.pi)
axes[1].set_title("Phase")
for axis in axes:
    axis.set_axis_off()
fig.tight_layout()
plt.show()
